In [1]:
import pandas as pd
import numpy as np

# pip install category-encoders
import category_encoders as ce

In [2]:
import sys
print(sys.executable)

c:\Users\Gayani\Advanced Data Analytics\AdvancedDataAnalytics\.venv\Scripts\python.exe


In [3]:
# the first column of the file is just an unnamed row number => use it as the index
df = pd.read_csv("flight_price_Clean_Dataset.csv", index_col=0)
df

,airline,flight,source_city,departure_time,stops,arrival_time,destination_city,class,duration,days_left,price
0,SpiceJet,SG-8709,Delhi,Evening,zero,Night,Mumbai,Economy,2.17,1,5953
1,SpiceJet,SG-8157,Delhi,Early_Morning,zero,Morning,Mumbai,Economy,2.33,1,5953
2,AirAsia,I5-764,Delhi,Early_Morning,zero,Early_Morning,Mumbai,Economy,2.17,1,5956
3,Vistara,UK-995,Delhi,Morning,zero,Afternoon,Mumbai,Economy,2.25,1,5955
4,Vistara,UK-963,Delhi,Morning,zero,Morning,Mumbai,Economy,2.33,1,5955
...,...,...,...,...,...,...,...,...,...,...,...
300148,Vistara,UK-822,Chennai,Morning,one,Evening,Hyderabad,Business,10.08,49,69265
300149,Vistara,UK-826,Chennai,Afternoon,one,Night,Hyderabad,Business,10.42,49,77105
300150,Vistara,UK-832,Chennai,Early_Morning,one,Night,Hyderabad,Business,13.83,49,79099
300151,Vistara,UK-828,Chennai,Early_Morning,one,Evening,Hyderabad,Business,10.00,49,81585


In [4]:
# the flight column (for example UK-706) has over 1500 unique options
df['flight'].nunique()

1561

**Example 1: Target encoding (replace each unique flight with that flight's average price)**

In [5]:
# create a target encoder by using category encoders
encoder = ce.TargetEncoder(cols=["flight"])

# apply the encoder, flight is going to be encoded, price is our target variable
df_encoded_target = encoder.fit_transform(df['flight'], df['price'])

# insert the encoded flight averages back to the original data
df['flight_encoded'] = df_encoded_target

# the usual process is this => you save and use the original flight code
# => then you get the matching average from a separate file (which we created based on the data)
# => right before you use the ML model to predict the price => drop the original flight code
# df.drop("flight", axis=1)
df.head(5)

,airline,flight,source_city,departure_time,stops,arrival_time,destination_city,class,duration,days_left,price,flight_encoded
0,SpiceJet,SG-8709,Delhi,Evening,zero,Night,Mumbai,Economy,2.17,1,5953,4740.222372
1,SpiceJet,SG-8157,Delhi,Early_Morning,zero,Morning,Mumbai,Economy,2.33,1,5953,5072.414608
2,AirAsia,I5-764,Delhi,Early_Morning,zero,Early_Morning,Mumbai,Economy,2.17,1,5956,3909.345184
3,Vistara,UK-995,Delhi,Morning,zero,Afternoon,Mumbai,Economy,2.25,1,5955,31270.540039
4,Vistara,UK-963,Delhi,Morning,zero,Morning,Mumbai,Economy,2.33,1,5955,33805.822695


In [6]:
# based on target encoding, average price for UK-706
df_test = df[df['flight'] == "UK-706"]
df_test['price'].mean().item()

# price is our target variable
# for each category in flight
# we replace the text category with its average of the price value

32726.911901081916

**If data overfits easily with ML model and the target encoded data (average), you can use LeaveOneOutEncoder instead to reduce overfitting (adds a bit of variance)**

In [7]:
# create a leave one out encoder by using category encoders
encoder = ce.LeaveOneOutEncoder(cols=["flight"])

# apply the encoder, flight is going to be encoded, price is our target variable
df_encoded_target = encoder.fit_transform(df['flight'], df['price'])

# insert the encoded flight averages back to the original data
df['flight_LOO_encoded'] = df_encoded_target

# the usual process is this => you save and use the original flight code
# => then you get the matching average from a separate file (which we created based on the data)
# => right before you use the ML model to predict the price => drop the original flight code
# df.drop("flight", axis=1)
df.head(5)

,airline,flight,source_city,departure_time,stops,arrival_time,destination_city,class,duration,days_left,price,flight_encoded,flight_LOO_encoded
0,SpiceJet,SG-8709,Delhi,Evening,zero,Night,Mumbai,Economy,2.17,1,5953,4740.222372,4730.833333
1,SpiceJet,SG-8157,Delhi,Early_Morning,zero,Morning,Mumbai,Economy,2.33,1,5953,5072.414608,5065.400000
2,AirAsia,I5-764,Delhi,Early_Morning,zero,Early_Morning,Mumbai,Economy,2.17,1,5956,3909.345184,3881.216495
3,Vistara,UK-995,Delhi,Morning,zero,Afternoon,Mumbai,Economy,2.25,1,5955,31270.540039,31295.286413
4,Vistara,UK-963,Delhi,Morning,zero,Morning,Mumbai,Economy,2.33,1,5955,33805.822695,33834.068966


In [8]:
# Leave One Out-encoder is almost the same as 
# Target Encoding, BUT for each row in the dataset
# and the average target calculation, THAT ONE PARTICULAR
# row is excluded from the calculation
category_row_prices = {
    2,
    5,
    #7,
    2,
    5,
    1
}

# quick average calculation
average = sum(category_row_prices) / len(category_row_prices)
average

2.6666666666666665

**Example 2 - Frequency (count) encoder => replace each unique flight with the frequency/count of that flight (probably measures how popular a flight is, if that's important for your ML model)**

In [9]:
# quick check, top 5 value counts of flight
df['flight'].value_counts().head(5)

flight
UK-706    3235
UK-772    2741
UK-720    2650
UK-836    2542
UK-822    2468
Name: count, dtype: int64

In [10]:
# create a count encoder by using category encoders
encoder = ce.CountEncoder(cols=["flight"])

# apply the encoder, flight is going to be encoded
df_encoded_target = encoder.fit_transform(df['flight'])

# insert the encoded flight counts back to the original data
df['flight_frequency_encoded'] = df_encoded_target

# the usual process is this => you save and use the original flight code
# => then you get the matching count from a separate file (which we created based on the data)
# => right before you use the ML model to predict the price => drop the original flight code
# df.drop("flight", axis=1)
df[['flight', 'price', 'flight_frequency_encoded']].head(5)

# REMEMBER: if you drop the original flight and after that remove all duplicates
# you will most likely have less rows in the data because of that

# the frequency/count approach is probably most effective when
# the popularity of some certain flight has a meaning in ML model
# for flight prices, popular flights (many tickets) can have different prices than rare ones
# (etc. depends on context/situation however)

,flight,price,flight_frequency_encoded
0,SG-8709,5953,133
1,SG-8157,5953,131
2,I5-764,5956,98
3,UK-995,5955,1024
4,UK-963,5955,987


**Example 3 - Hash encoding / feature hashing (or hashing trick)**

In [11]:
# let's save this for studying the hashing buckets after the operation
# HashingEncoder will remove the original category after being used
df['flight_original'] = df['flight']

In [12]:
# create a hashing encoder by using category encoders
encoder = ce.HashingEncoder(cols=['flight'], n_components=6)

# apply the encoder, flight is going to be hashed into 6 buckets
df_encoded_hash = encoder.fit_transform(df)

# we have now simplified 1500+ categories into 6
# consult Google/your favorite generative AI for ideas on how 
# to evaluate the quality of the hashing and the amount of buckets
df_encoded_hash

,col_0,col_1,col_2,col_3,col_4,col_5,airline,source_city,departure_time,stops,arrival_time,destination_city,class,duration,days_left,price,flight_encoded,flight_LOO_encoded,flight_frequency_encoded,flight_original
0,0,1,0,0,0,0,SpiceJet,Delhi,Evening,zero,Night,Mumbai,Economy,2.17,1,5953,4740.222372,4730.833333,133,SG-8709
1,0,0,1,0,0,0,SpiceJet,Delhi,Early_Morning,zero,Morning,Mumbai,Economy,2.33,1,5953,5072.414608,5065.400000,131,SG-8157
2,0,0,0,0,0,1,AirAsia,Delhi,Early_Morning,zero,Early_Morning,Mumbai,Economy,2.17,1,5956,3909.345184,3881.216495,98,I5-764
3,1,0,0,0,0,0,Vistara,Delhi,Morning,zero,Afternoon,Mumbai,Economy,2.25,1,5955,31270.540039,31295.286413,1024,UK-995
4,0,1,0,0,0,0,Vistara,Delhi,Morning,zero,Morning,Mumbai,Economy,2.33,1,5955,33805.822695,33834.068966,987,UK-963
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
300148,0,1,0,0,0,0,Vistara,Chennai,Morning,one,Evening,Hyderabad,Business,10.08,49,69265,27586.745543,27569.851236,2468,UK-822
300149,0,1,0,0,0,0,Vistara,Chennai,Afternoon,one,Night,Hyderabad,Business,10.42,49,77105,26276.531478,26254.754070,2335,UK-826
300150,0,0,0,0,1,0,Vistara,Chennai,Early_Morning,one,Night,Hyderabad,Business,13.83,49,79099,31499.719218,31479.910945,2404,UK-832
300151,0,0,0,0,0,1,Vistara,Chennai,Early_Morning,one,Evening,Hyderabad,Business,10.00,49,81585,28283.469262,28261.615416,2440,UK-828


In [13]:
df_encoded_hash.columns

Index(['col_0', 'col_1', 'col_2', 'col_3', 'col_4', 'col_5', 'airline',
       'source_city', 'departure_time', 'stops', 'arrival_time',
       'destination_city', 'class', 'duration', 'days_left', 'price',
       'flight_encoded', 'flight_LOO_encoded', 'flight_frequency_encoded',
       'flight_original'],
      dtype='object')

In [14]:
# let's quickly see what kind of flights were placed in the first "cluster" / bucket
df_encoded_hash.query('col_0 == 1')['flight_original']

3          UK-995
15        6E-6278
28        SG-8169
37         AI-453
45         SG-339
           ...   
300111     AI-545
300114     AI-569
300141     AI-430
300145     AI-430
300147     AI-569
Name: flight_original, Length: 48089, dtype: object

In [15]:
# there's still plenty of different encoders in the category encoders
# -module, feel free to experiment!